# 🏔️ Lakehouse Exploration — Medallion Architecture

This notebook explores all three medallion layers using PySpark + Delta Lake.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum as _sum, avg

spark = (
    SparkSession.builder
    .appName('LakehouseExplorer')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.hadoop.fs.s3a.endpoint', 'http://minio:9000')
    .config('spark.hadoop.fs.s3a.access.key', 'minioadmin')
    .config('spark.hadoop.fs.s3a.secret.key', 'minioadmin')
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.aws.credentials.provider',
            'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('Spark ready ✅')

## 🥉 Bronze Layer — Raw events

In [ ]:
bronze_orders = spark.read.format('delta').load('s3a://bronze/orders')
print(f'Bronze orders count: {bronze_orders.count()}')
bronze_orders.printSchema()
bronze_orders.show(5, truncate=80)

In [ ]:
bronze_clicks = spark.read.format('delta').load('s3a://bronze/clicks')
print(f'Bronze clicks count: {bronze_clicks.count()}')
bronze_clicks.show(5, truncate=80)

## 🥉 Bronze Delta Table History

In [ ]:
from delta.tables import DeltaTable
dt = DeltaTable.forPath(spark, 's3a://bronze/orders')
dt.history().select('version','timestamp','operation','operationMetrics').show(10, truncate=False)

## ⏪ Time Travel — Read an older version

In [ ]:
# Read version 0 of bronze orders
v0 = spark.read.format('delta').option('versionAsOf', 0).load('s3a://bronze/orders')
print(f'Version 0 row count: {v0.count()}')
v0.show(3, truncate=60)

## 🥈 Silver Layer — Cleaned & typed

In [ ]:
silver_orders = spark.read.format('delta').load('s3a://silver/orders')
print(f'Silver orders count: {silver_orders.count()}')
silver_orders.printSchema()

In [ ]:
# Status distribution
silver_orders.groupBy('status').count().orderBy('count', ascending=False).show()

In [ ]:
# Revenue by category
(
    silver_orders
    .filter(col('status') == 'completed')
    .groupBy('category')
    .agg(
        count('order_id').alias('orders'),
        _sum('total_amount').alias('revenue'),
        avg('total_amount').alias('avg_value')
    )
    .orderBy('revenue', ascending=False)
    .show()
)

## 🥇 Gold Layer — Aggregations

In [ ]:
daily_rev = spark.read.format('delta').load('s3a://gold/daily_revenue')
daily_rev.orderBy('date', ascending=False).show(10)

In [ ]:
product_metrics = spark.read.format('delta').load('s3a://gold/product_metrics')
print(f'Products tracked: {product_metrics.count()}')
product_metrics.orderBy('total_revenue', ascending=False).show(10)

In [ ]:
user_metrics = spark.read.format('delta').load('s3a://gold/user_metrics')
print(f'Users tracked: {user_metrics.count()}')
user_metrics.orderBy('total_spend', ascending=False).show(10)

In [ ]:
funnel = spark.read.format('delta').load('s3a://gold/funnel')
funnel.orderBy('visits', ascending=False).show()

## 📊 Quick visualisation — Revenue by day

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = daily_rev.orderBy('date').toPandas()
if not df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].bar(df['date'].astype(str), df['revenue'])
    axes[0].set_title('Daily Revenue')
    axes[0].tick_params(axis='x', rotation=45)
    axes[1].bar(df['date'].astype(str), df['order_count'])
    axes[1].set_title('Daily Order Count')
    axes[1].tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print('No data yet — wait for the batch job to complete')